<a href="https://colab.research.google.com/github/MeherNaaz19/NLP/blob/main/2303A51902_B09_A_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv('/content/tweets[1].csv')

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from string import punctuation
import nltk
import string
from nltk.stem import WordNetLemmatizer # Import WordNetLemmatizer
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet') # Download the wordnet corpus
nltk.download('punkt_tab') # Download punkt_tab
lemmatizer = WordNetLemmatizer() # Initialize the lemmatizer
stop_words = set(stopwords.words('english'))
def preprocess_text(text):
    text=text.lower()
    text=text.translate(str.maketrans('', '', string.punctuation)) # Remove punctuation
    text=re.sub(r'[^a-zA-Z0-9\s]', '', text)
    tokens=word_tokenize(text)#tokenization
    stop_words=set(stopwords.words('english'))#stop words
    tokens=[word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]  # Lemmatization
    return " ".join(tokens)
df["cleaned_text"]=df["text"].astype(str).apply(preprocess_text)
print(df)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


          id  keyword                 location  \
0          0   ablaze                      NaN   
1          1   ablaze                      NaN   
2          2   ablaze            New York City   
3          3   ablaze           Morgantown, WV   
4          4   ablaze                      NaN   
...      ...      ...                      ...   
11365  11365  wrecked  Blue State in a red sea   
11366  11366  wrecked               arohaonces   
11367  11367  wrecked                       🇵🇭   
11368  11368  wrecked           auroraborealis   
11369  11369  wrecked                      NaN   

                                                    text  target  \
0      Communal violence in Bhainsa, Telangana. "Ston...       1   
1      Telangana: Section 144 has been imposed in Bha...       1   
2      Arsonist sets cars ablaze at dealership https:...       1   
3      Arsonist sets cars ablaze at dealership https:...       1   
4      "Lord Jesus, your love brings freedom and pard...   

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer(ngram_range=(1,2),max_features=20000)
X=vectorizer.fit_transform(df["cleaned_text"])
y=df["target"]

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout,Embedding,Conv1D,GlobalMaxPooling1D,LSTM,Bidirectional
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
mlp = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
mlp.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
mlp.fit(X_train.toarray(),y_train,validation_data=(X_test.toarray(),y_test),epochs=5,batch_size=64)

Epoch 1/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - accuracy: 0.8032 - loss: 0.5225 - val_accuracy: 0.8923 - val_loss: 0.2809
Epoch 2/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9278 - loss: 0.2047 - val_accuracy: 0.9028 - val_loss: 0.2584
Epoch 3/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9762 - loss: 0.0810 - val_accuracy: 0.8962 - val_loss: 0.3061
Epoch 4/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9894 - loss: 0.0374 - val_accuracy: 0.8997 - val_loss: 0.3628
Epoch 5/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9943 - loss: 0.0188 - val_accuracy: 0.8953 - val_loss: 0.4164


In [ ]:
loss,accuracy = mlp.evaluate(X_test.toarray(), y_test)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8995 - loss: 0.4034
Test Loss: 0.4164
Test Accuracy: 0.8953


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
tokenizer=Tokenizer(num_words=10000)
tokenizer.fit_on_texts(df["cleaned_text"])
X_seq=tokenizer.texts_to_sequences(df["cleaned_text"])
X_pad=pad_sequences(X_seq,maxlen=100)
X_train_seq,X_test_seq,y_train,y_test=train_test_split(X_pad,df["target"],test_size=0.2,random_state=42)
vocab_size=len(tokenizer.word_index)+1
cnn=Sequential([
    Embedding(vocab_size,128,input_length=100),
    Conv1D(128,5,activation="relu"),
    GlobalMaxPooling1D(),
    Dense(64,activation="relu"),
    Dropout(0.3),
    Dense(1,activation="sigmoid")
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
cnn.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
cnn.fit(X_train.toarray(),y_train,validation_data=(X_test.toarray(),y_test),epochs=5,batch_size=64)

Epoch 1/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 81s 461ms/step - accuracy: 0.7871 - loss: 0.5371 - val_accuracy: 0.8259 - val_loss: 0.4659
Epoch 2/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 58s 387ms/step - accuracy: 0.8098 - loss: 0.4933 - val_accuracy: 0.8259 - val_loss: 0.4638
Epoch 3/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 81s 382ms/step - accuracy: 0.8122 - loss: 0.4847 - val_accuracy: 0.8259 - val_loss: 0.4610
Epoch 4/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 83s 387ms/step - accuracy: 0.8114 - loss: 0.4896 - val_accuracy: 0.8259 - val_loss: 0.4644
Epoch 5/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 82s 386ms/step - accuracy: 0.8075 - loss: 0.4941 - val_accuracy: 0.8259 - val_loss: 0.4606


In [ ]:
lstm=Sequential([
    Embedding(vocab_size,128,input_length=100),
    Bidirectional(LSTM(64,return_sequences=False)),
    Dropout(0.3),
    Dense(1,activation="sigmoid")
])
lstm.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
lstm.fit(X_train_seq,y_train,validation_data=(X_test_seq,y_test),epochs=5,batch_size=64)

Epoch 1/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - accuracy: 0.7924 - loss: 0.4751 - val_accuracy: 0.8997 - val_loss: 0.2676
Epoch 2/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.9278 - loss: 0.1923 - val_accuracy: 0.9050 - val_loss: 0.2588
Epoch 3/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9652 - loss: 0.1074 - val_accuracy: 0.9011 - val_loss: 0.2749
Epoch 4/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.9838 - loss: 0.0576 - val_accuracy: 0.8980 - val_loss: 0.3546
Epoch 5/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.9895 - loss: 0.0407 - val_accuracy: 0.8989 - val_loss: 0.4011


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
# Make predictions
mlp_y_pred = (mlp.predict(X_test.toarray()) > 0.5).astype("int32")
cnn_y_pred = (cnn.predict(X_test_seq) > 0.5).astype("int32")
lstm_y_pred = (lstm.predict(X_test_seq) > 0.5).astype("int32")
# Generate and print classification reports
print("MLP Model Classification Report:")
print(classification_report(y_test, mlp_y_pred))
print("\nCNN Model Classification Report:")
print(classification_report(y_test, cnn_y_pred))
print("\nBi-LSTM Model Classification Report:")
print(classification_report(y_test, lstm_y_pred))
# Generate and print confusion matrices
print("MLP Model Confusion Matrix:")
print(confusion_matrix(y_test, mlp_y_pred))
print("\nCNN Model Confusion Matrix:")
print(confusion_matrix(y_test, cnn_y_pred))
print("\nBi-LSTM Model Confusion Matrix:")
print(confusion_matrix(y_test, lstm_y_pred))

72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
MLP Model Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.96      0.94      1878
           1       0.75      0.60      0.67       396

    accuracy                           0.90      2274
   macro avg       0.83      0.78      0.80      2274
weighted avg       0.89      0.90      0.89      2274


CNN Model Classification Report:
              precision    recall  f1-score   support

           0       0.83      1.00      0.90      1878
           1       0.00      0.00      0.00       396

    accuracy                           0.83      2274
   macro avg       0.41      0.50      0.45      2274
weighted avg       0.68      0.83      0.75      2274


Bi-LSTM Model Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.96      0.94      1878
           1   

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_y_pred = lr_model.predict(X_test)
print("Logistic Regression Model Classification Report:")
print(classification_report(y_test, lr_y_pred))
print("\nLogistic Regression Model Confusion Matrix:")
print(confusion_matrix(y_test, lr_y_pred))

Logistic Regression Model Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.99      0.93      1878
           1       0.90      0.29      0.44       396

    accuracy                           0.87      2274
   macro avg       0.88      0.64      0.68      2274
weighted avg       0.87      0.87      0.84      2274


Logistic Regression Model Confusion Matrix:
[[1865   13]
 [ 280  116]]


In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
svm_model = SVC()
svm_model.fit(X_train, y_train)
svm_y_pred = svm_model.predict(X_test)
print("SVM Model Classification Report:")
print(classification_report(y_test, svm_y_pred))
print("\nSVM Model Confusion Matrix:")
print(confusion_matrix(y_test, svm_y_pred))

SVM Model Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.99      0.94      1878
           1       0.93      0.43      0.59       396

    accuracy                           0.89      2274
   macro avg       0.91      0.71      0.76      2274
weighted avg       0.90      0.89      0.88      2274


SVM Model Confusion Matrix:
[[1865   13]
 [ 226  170]]


Accuracy:
MLP and Bi-LSTM: ~90%
SVM: ~89%
Logistic Regression: ~87%
CNN: ~83%
Recall:
MLP and Bi-LSTM: ~60%
SVM: 43%
Logistic Regression: 29%
CNN: 0%
Finding: Deep learning models (MLP and Bi-LSTM) were significantly better at identifying actual disaster tweets compared to Logistic Regression and SVM.
Precision:
Logistic Regression: 90%
SVM: 93%
MLP and Bi-LSTM: ~75-77%
CNN: 0%
Finding: Logistic Regression and SVM had higher precision, meaning when they predicted a tweet was a disaster tweet, they were more often correct than the deep learning models. However, as seen by recall, they made fewer positive predictions overall.
F1-score :
MLP and Bi-LSTM: ~67%
SVM: 59%
Logistic Regression: 44%
CNN: 0%
Finding: The F1-score, which balances precision and recall, shows that the MLP and Bi-LSTM models achieved the best balance for the minority class. SVM also had a decent F1-score, outperforming Logistic Regression.

Based on these metrics, particularly the F1-score and recall for the minority class, the deep learning models (MLP and Bi-LSTM) and SVM generally outperformed Logistic Regression. Between deep learning and SVM, MLP and Bi-LSTM show higher recall and F1-score for the minority class, indicating they are better at identifying disaster tweets, although SVM has higher precision for this class.

Considering the F1-score for the minority class as a key metric, the MLP and Bi-LSTM architectures worked best in this scenario.

both MLP and Bi-LSTM deep learning models, as well as the SVM model, showed good performance on this task, outperforming Logistic Regression.